# M4.0a Primer: The Magnetic Field, Magnetic Forces and the Biot-Savart Law

Book pointer: chapter 11, sections 11.1 (the magnetic field) and 11.2 (magnetic forces).

First of three primers for module 4. Modules 1 to 3 dealt with charges at rest. This module starts charges moving, and moving charges both feel and produce a second field, the magnetic field $\mathbf{B}$. This primer defines $\mathbf{B}$ through the force it exerts, works out how charged particles move in it, derives the Biot-Savart law that says how currents produce it, and does the two classic cases (straight wire, circular loop) by hand. Primer 00b takes up flux, Ampere's law and the vector potential; primer 00c takes up magnetized matter. Every result here is reproduced in code with `emlib.biot_savart` and `emlib.integrators`, the two library modules that the main notebooks of this module call as finished tools.

## 1. Warm-up

Try on paper first.

1. Write the force on a charge $q$ moving with velocity $\mathbf{v}$ through electric and magnetic fields. Which part can change the particle's speed?
2. A proton moves along $+x$ through a magnetic field along $+z$. Which way is the force? Which way for an electron?
3. Why does a charged particle in a uniform magnetic field move in a circle, and what sets the radius and the period?
4. What is the force per unit length between two long parallel wires carrying currents in the same direction, and is it attractive or repulsive?
5. State the Biot-Savart law. What plays the role that $q$ and $\hat{\mathbf{R}}/R^2$ played in Coulomb's law?
6. Derive, or at least set up, the magnetic field at the centre of a circular loop of radius $R$ carrying current $I$.
7. Far from a small current loop, what does its field look like, and what single vector characterises it?

## 2. The ideas

**Magnetism is a force between moving charges.** Two charges at rest push on each other with the Coulomb force. Set them both moving and a second force appears, perpendicular to the velocities, proportional to both of them, and far weaker at ordinary speeds. Experimentally it is easiest to see with currents, which are enormous numbers of charges moving together while the wire as a whole stays neutral: two parallel wires attract when their currents run the same way and repel when they run opposite, and a compass needle near a wire turns to point around the wire rather than towards it. Everything in this chapter and the next is the description of that force, organised in the same way as electrostatics: a field that the sources produce, and a force law that says what the field does to a charge placed in it.

**The magnetic field is defined by the force it exerts.** The electric field was defined as force per unit charge. The magnetic field cannot be defined that simply, because the magnetic force depends on the velocity as well as the charge. What experiment shows is that the force on a charge $q$ moving with velocity $\mathbf{v}$ is always perpendicular to $\mathbf{v}$, proportional to $q$ and to $|\mathbf{v}|$, and vanishes along one particular direction at each point. All of that is captured by a single vector $\mathbf{B}$ and the cross product: $\mathbf{F} = q\,\mathbf{v}\times\mathbf{B}$. The direction along which the force vanishes is the direction of $\mathbf{B}$, and the force is largest when $\mathbf{v}$ is perpendicular to it. Adding the electric part gives the Lorentz force, $\mathbf{F} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$, which is the complete law of force on a point charge in classical electromagnetism. The unit of $\mathbf{B}$ is the tesla: one tesla exerts one newton on one coulomb moving at one metre per second across the field. The Earth's field is about $5\times10^{-5}\,$T, a fridge magnet about $10^{-2}\,$T, the bore of an MRI scanner $1.5$ to $3\,$T.

**Magnetic forces do no work.** Because $\mathbf{v}\times\mathbf{B}$ is perpendicular to $\mathbf{v}$, the power $\mathbf{F}\cdot\mathbf{v}$ delivered by a magnetic field is identically zero. A magnetic field can bend a trajectory but never speed a particle up or slow it down. This single fact explains the shape of every orbit in this primer: in a uniform field a particle keeps its speed and turns at a constant rate, so it moves in a circle, or in a helix if it also drifts along the field. It also sets up a puzzle for later: a wire does gain energy when it is lifted by a magnet, and the resolution (it is the electric field inside the wire that does the work, and the magnetic field only redirects it) is in section 3.1.

**Currents produce magnetic fields.** The Biot-Savart law is the magnetic counterpart of Coulomb's law. A short piece of wire of length $d\mathbf{l}$ carrying current $I$ produces at a displacement $\mathbf{R}$ the field $d\mathbf{B} = \dfrac{\mu_0}{4\pi}\dfrac{I\,d\mathbf{l}\times\hat{\mathbf{R}}}{R^2}$. The inverse square is the same as Coulomb; the cross product is new and makes the field circulate around the current instead of pointing away from it. The constant $\mu_0 = 4\pi\times10^{-7}\,$T m/A plays the role of $1/\epsilon_0$. Superposition holds exactly as it did for charges, so the field of any circuit is the sum (the integral) of the contributions of its pieces, and on a computer that is a vectorised sum over segments, which is what `emlib.biot_savart.wire_B` does.

**Field lines close on themselves.** Electric field lines start on positive charge and end on negative charge. Magnetic field lines do neither: there is no magnetic charge, so the lines of $\mathbf{B}$ have no beginning and no end, and around a wire they are closed circles. Primer 00b turns that picture into the law $\nabla\cdot\mathbf{B} = 0$ and into Ampere's law, which is the magnetic counterpart of Gauss' law: the circulation of $\mathbf{B}$ around a closed loop counts the current threading it, just as the flux of $\mathbf{E}$ through a closed surface counts the charge inside.

**A current loop is a magnetic dipole.** Seen from far away, any small loop of current produces the same field as a point magnetic dipole, with moment $\mathbf{m} = I\mathbf{A}$, current times vector area. The field has the same shape as the electric dipole field of primer M1.0e, with $\mathbf{m}$ in place of $\mathbf{p}$ and $\mu_0/4\pi$ in place of $1/4\pi\epsilon_0$. A loop placed in an external field feels the torque $\mathbf{m}\times\mathbf{B}$ that turns it to align with the field; that torque is what moves the needle of a galvanometer and the rotor of a motor, and the moment per unit volume of a material is the magnetization of chapter 13.

**What the main notebooks do.** Notebook 01 builds loops and Helmholtz pairs from segments, sums the Biot-Savart contributions on a grid, draws field maps, and checks the on-axis closed form derived in section 3.6. Notebook 02 does the same for a solenoid and tests Ampere's law numerically. Notebook 03 integrates the Lorentz force for a particle with the Boris pusher, which section 3.3 introduces, and compares it with Runge-Kutta. Everything they rely on is derived below.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import biot_savart as bs
from emlib import integrators as it
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)
sci = lambda a: np.array2string(np.asarray(a, dtype=float), formatter={"float_kind": lambda x: f"{x:+.4e}"})   # print small numbers legibly

e = 1.602e-19       # C
m_p = 1.673e-27     # kg
m_e = 9.109e-31     # kg

### 3.1 The Lorentz force

$$\mathbf{F} = q\,(\mathbf{E} + \mathbf{v}\times\mathbf{B}).$$

The electric part is familiar. The magnetic part has three properties to keep in mind. It is perpendicular to both $\mathbf{v}$ and $\mathbf{B}$, with the direction given by the right-hand rule for the cross product (fingers along $\mathbf{v}$, curl towards $\mathbf{B}$, thumb gives $\mathbf{v}\times\mathbf{B}$) and then flipped if $q$ is negative. Its magnitude is $|q||\mathbf{v}||\mathbf{B}|\sin\theta$, largest across the field and zero along it. And it does no work: $\mathbf{F}_{\text{mag}}\cdot\mathbf{v} = q\,(\mathbf{v}\times\mathbf{B})\cdot\mathbf{v} = 0$ because a cross product is perpendicular to each of its factors (primer M1.0a, section 3.3). So the kinetic energy of a particle changes only through $\mathbf{E}$; the magnetic field only steers.

**Worked example 1.** A proton ($q = +e$) moves at $v = 10^5\,$m/s along $+x$ through $\mathbf{B} = 0.01\,\hat{\mathbf{z}}\,$T. $\mathbf{v}\times\mathbf{B} = v B\,(\hat{\mathbf{x}}\times\hat{\mathbf{z}}) = -vB\,\hat{\mathbf{y}}$. Force $= 1.602\times10^{-19}\cdot10^5\cdot0.01 = 1.602\times10^{-16}\,$N along $-y$. An electron with the same velocity feels the same magnitude along $+y$. For scale, gravity on the proton is $m_pg = 1.64\times10^{-26}\,$N, ten billion times smaller: gravity is ignored in everything that follows.

**Force on a current.** A wire carries no net charge, but the charges that make up its current are moving, and each feels $q\mathbf{v}\times\mathbf{B}$. Let the wire have cross-section $A$, carrier density $n$ (carriers per volume), carrier charge $q$ and drift velocity $\mathbf{v}_d$ along the wire. A piece of length $dl$ contains $nA\,dl$ carriers, so the force on it is $nA\,dl\;q\,\mathbf{v}_d\times\mathbf{B}$. The current is $I = nqv_dA$ (charge crossing a section per second), and $\mathbf{v}_d\,dl$ has the direction of the current with magnitude $v_d\,dl$, so
$$d\mathbf{F} = I\,d\mathbf{l}\times\mathbf{B},$$
with $d\mathbf{l}$ pointing along the current. The carriers push on the lattice of the wire through collisions, and that is how the force reaches the wire as a whole. (And here is the resolution of the no-work puzzle: when the wire moves, the magnetic force on the carriers acquires a component along the wire that opposes the current, and the battery's electric field does the work against it. The magnetic field redirects energy from the battery into the motion of the wire; it supplies none itself.)

For a straight wire of length $L$ in a uniform field the integral is immediate: $\mathbf{F} = I\mathbf{L}\times\mathbf{B}$, magnitude $ILB\sin\theta$.

**Worked example 2.** A $0.5\,$m wire carrying $10\,$A at right angles to a $0.2\,$T field: $F = 10\cdot0.5\cdot0.2 = 1\,$N. At $30^\circ$ to the field: $0.5\,$N. Along the field: zero.

**Two parallel wires.** Wire 1 along $z$ carries $I_1$. Its field at distance $d$ is $B = \mu_0I_1/2\pi d$ (derived in section 3.6), circulating around it. Wire 2, parallel at distance $d$, carrying $I_2$ in the same direction, sits in that field, and $I_2\,d\mathbf{l}\times\mathbf{B}$ points towards wire 1. So the force per unit length is
$$\frac{F}{L} = \frac{\mu_0I_1I_2}{2\pi d}, \qquad \text{attractive for parallel currents, repulsive for antiparallel.}$$
Reverse one current and $d\mathbf{l}$ flips, so the force flips. This formula used to define the ampere: two wires one metre apart carrying one ampere each feel $2\times10^{-7}\,$N per metre.

**Worked example 3.** Two wires $1\,$cm apart each carrying $100\,$A: $F/L = 2\times10^{-7}\cdot100\cdot100/0.01 = 0.2\,$N/m. The code builds wire 1 from segments, evaluates its field on a one-metre stretch of wire 2 with `wire_B`, and sums $I_2\,d\mathbf{l}\times\mathbf{B}$.

In [ ]:
# Worked example 1: the force on a proton.
v = np.array([1e5, 0.0, 0.0]); B = np.array([0.0, 0.0, 0.01])
F = e * np.cross(v, B)
print("force on the proton:", sci(F), "N   magnitude", f"{np.linalg.norm(F):.3e}", "   gravity:", f"{m_p * 9.81:.3e}")
print("power delivered F . v =", F @ v)

# Worked example 3: force per length between two parallel wires 1 cm apart, 100 A each.
I1, I2, d = 100.0, 100.0, 0.01
mid1, dl1 = bs.segments(bs.straight_wire([0, 0, -50], [0, 0, 50], 100000), closed=False)   # wire 1 on the z axis
path2 = bs.straight_wire([d, 0, -0.5], [d, 0, 0.5], 200)                                    # one metre of wire 2
mid2, dl2 = bs.segments(path2, closed=False)
B_on_2 = bs.wire_B(mid1, dl1, I1, mid2)
F_on_2 = I2 * np.cross(dl2, B_on_2).sum(axis=0)
print(f"force on 1 m of wire 2: {F_on_2} N    formula mu0 I1 I2 / (2 pi d) = {MU0 * I1 * I2 / (2 * np.pi * d):.4f} N/m   (negative x: towards wire 1)")
path2r = path2[::-1]; mid2r, dl2r = bs.segments(path2r, closed=False)
print("with the current in wire 2 reversed:", I2 * np.cross(dl2r, bs.wire_B(mid1, dl1, I1, mid2r)).sum(axis=0), "N  (repulsive)")

### 3.2 The Hall effect: seeing the carriers

The force $I\,d\mathbf{l}\times\mathbf{B}$ acts on the carriers, not on the wire, and the carriers respond before the lattice does. Take a flat strip of width $w$ and thickness $t$ carrying current $I$ along $x$, in a field $\mathbf{B} = B\hat{\mathbf{z}}$ normal to the strip. The carriers drift along the strip at $v_d$ and are pushed sideways, across the width, by $q\mathbf{v}_d\times\mathbf{B}$. They pile up on one edge until the electric field of that pile-up, $E_H$ across the strip, balances the magnetic push: $qE_H = qv_dB$, so $E_H = v_dB$, and a voltage $V_H = E_Hw = v_dBw$ appears between the edges. With $I = nqv_dA = nqv_d\,wt$ this is
$$V_H = \frac{IB}{nqt}.$$
Two things make this more than a curiosity. First, the sign of $V_H$ tells which edge the carriers pile on, hence the sign of $q$: in most metals the carriers are electrons, and the Hall voltage says so. Second, the size of $V_H$ gives the carrier density $n$, which no electrostatic measurement can. In reverse, with $n$ known, a Hall probe is the standard instrument for measuring $B$.

**Worked example 4.** A copper strip $1\,$cm wide and $1\,$mm thick carries $10\,$A in a $1\,$T field. Copper has about one conduction electron per atom, $n = 8.5\times10^{28}\,$m$^{-3}$. Drift speed: $v_d = I/nqA = 10/(8.5\times10^{28}\cdot1.602\times10^{-19}\cdot10^{-5}) = 7.3\times10^{-5}\,$m/s, a tenth of a millimetre per second. Hall field $E_H = v_dB = 7.3\times10^{-5}\,$V/m and Hall voltage $V_H = E_Hw = 7.3\times10^{-7}\,$V, less than a microvolt: the reason Hall probes use thin semiconductors, where $n$ is a million times smaller and $V_H$ a million times larger.

In [ ]:
n_cu, w, t, I, B0 = 8.5e28, 0.01, 1e-3, 10.0, 1.0
v_d = I / (n_cu * e * w * t)
E_H = v_d * B0
print(f"drift speed {v_d:.3e} m/s   Hall field {E_H:.3e} V/m   Hall voltage V_H = E_H w = {E_H * w:.3e} V   formula I B / (n q t) = {I * B0 / (n_cu * e * t):.3e} V")
print(f"the same strip in silicon with n = 1e22 m^-3 would give V_H = {I * B0 / (1e22 * e * t):.3e} V")

### 3.3 Motion in a uniform magnetic field

Put a particle of charge $q$ and mass $m$ in a uniform field $\mathbf{B} = B\hat{\mathbf{z}}$ with no electric field, and start it with velocity $\mathbf{v}$ perpendicular to $\mathbf{B}$. The force $q\mathbf{v}\times\mathbf{B}$ has magnitude $qvB$, is perpendicular to $\mathbf{v}$, and cannot change the speed. A force of constant magnitude always perpendicular to the velocity is exactly a centripetal force, so the motion is a circle. Setting $qvB = mv^2/r$ gives the radius and, from $T = 2\pi r/v$, the period:
$$r = \frac{mv}{|q|B} \quad\text{(the Larmor radius)}, \qquad \omega = \frac{2\pi}{T} = \frac{|q|B}{m} \quad\text{(the cyclotron frequency)}.$$
The frequency does not depend on the speed: a faster particle traces a bigger circle in the same time. That is the principle of the cyclotron, which accelerates particles with an alternating voltage at this fixed frequency while the field keeps turning them back. The radius is proportional to momentum, which is the principle of the mass spectrometer: particles of the same charge and speed but different masses land at different radii.

**Sense of rotation.** For a positive charge with $\mathbf{B}$ along $+z$, the force on $\mathbf{v} = v\hat{\mathbf{x}}$ is along $-y$ (worked example 1), so the particle turns clockwise when viewed from the $+z$ side, looking against the field. A negative charge turns counter-clockwise. The two senses are also the senses of the magnetic moments of the orbits: a circulating positive charge is a current loop whose moment points along $-\mathbf{B}$, against the field (section 3.5), which is the origin of diamagnetism in chapter 13.

**Helices.** If the velocity has a component $v_\parallel$ along $\mathbf{B}$, that component feels no force and stays constant, while the perpendicular part circles as before. The path is a helix of radius $mv_\perp/|q|B$ and pitch $v_\parallel T$.

**Worked example 5.** A proton at $10^5\,$m/s in $0.01\,$T: $\omega = eB/m_p = 1.602\times10^{-21}/1.673\times10^{-27} = 9.58\times10^5\,$rad/s, $T = 6.56\,\mu$s, $r = m_pv/eB = 1.673\times10^{-22}/1.602\times10^{-21} = 0.1044\,$m. An electron at the same speed in the same field: $\omega = 1.76\times10^9\,$rad/s, $T = 3.57\,$ns, $r = 56.9\,\mu$m. Same field, same speed, a circle $1836$ times smaller and $1836$ times faster, the ratio of the masses.

**Worked example 6.** Give the proton an extra $3\times10^4\,$m/s along $z$. The perpendicular motion is unchanged (radius $0.1044\,$m, period $6.56\,\mu$s) and the pitch of the helix is $v_\parallel T = 3\times10^4\cdot6.56\times10^{-6} = 0.197\,$m per turn.

**The Boris pusher.** Notebook 03 integrates these equations numerically. Ordinary integrators (Euler, Runge-Kutta) let the speed drift because every step is a straight-line extrapolation that leaves the circle. The Boris method instead rotates the velocity about $\mathbf{B}$ by exactly the angle $\omega\,\Delta t$ (to second order, and by an exact rotation at any step size), with the electric kick split into two halves before and after the rotation. The speed is then conserved to rounding error for a pure magnetic field however large the step, and the energy stays bounded for ever. `emlib.integrators.boris` implements it; `emlib.integrators.rk4` is there for comparison. The code below runs the proton of worked examples 5 and 6, measures the radius and the period from the trajectory, and draws the circle and the helix.

**Animation.** Four short parts, all computed with the Boris pusher in units where the cyclotron frequency is 1. First, a positive and a negative charge in a field out of the page: the force (green) stays perpendicular to the velocity (white), so each turns at constant speed, the positive one clockwise and the negative one counter-clockwise, and the speed readout never moves from 1. Second, a velocity component along $\mathbf{B}$ turns the circle into a helix whose measured pitch equals $2\pi v_\parallel/\omega$. Third, the crossed fields of section 3.4: a proton and an electron both start at rest and both drift down the page along $\mathbf{E}\times\mathbf{B}$ at the measured speed $E/B$. Fourth, the torque of section 3.5 turning a current loop until its moment points along $\mathbf{B}$.

In [ ]:
from emlib import show
show.animation("m4_magnetostatics", "ChargedParticleMotion")

In [ ]:
q, m = e, m_p
B0 = 0.01
omega = it.cyclotron_frequency(q, B0, m); T = 2 * np.pi / omega
print(f"proton:   omega = {omega:.3e} rad/s   T = {T:.3e} s   r = {it.larmor_radius(m, 1e5, q, B0):.4f} m")
print(f"electron: omega = {it.cyclotron_frequency(e, B0, m_e):.3e} rad/s   T = {2 * np.pi / it.cyclotron_frequency(e, B0, m_e):.3e} s   r = {it.larmor_radius(m_e, 1e5, e, B0):.3e} m")

n = 400
x, v = it.boris([0, 0, 0], [1e5, 0, 0], q, m, [0, 0, 0], [0, 0, B0], T / n, 2 * n)
centre = x[:n].mean(axis=0)
radii = np.linalg.norm(x - centre, axis=1)
print(f"\nBoris, 400 steps per period: radius from the orbit {radii.mean():.5f} m (spread {radii.std():.1e}),  centre at y = {centre[1]:+.4f} (clockwise for a positive charge)")
print(f"after one period the proton is {np.linalg.norm(x[n] - x[0]):.2e} m from its start;  speed drift {abs(np.linalg.norm(v[-1]) / 1e5 - 1):.1e}")

xh, vh = it.boris([0, 0, 0], [1e5, 0, 3e4], q, m, [0, 0, 0], [0, 0, B0], T / n, 3 * n)
print(f"helix: z advanced per turn = {xh[n, 2] - xh[0, 2]:.4f} m   v_par T = {3e4 * T:.4f} m")

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1); ax1.plot(x[:, 0], x[:, 1]); ax1.plot(*x[0, :2], "o", color="C3", label="start, v along +x")
ax1.set_aspect("equal"); ax1.set_xlabel("x [m]"); ax1.set_ylabel("y [m]"); ax1.legend(); ax1.set_title("proton in B = 0.01 T along z: a circle, clockwise seen from +z")
ax2 = fig.add_subplot(1, 2, 2, projection="3d"); ax2.plot(xh[:, 0], xh[:, 1], xh[:, 2]); ax2.set_title("with v_z = 3e4 m/s: a helix"); ax2.set_xlabel("x"); ax2.set_ylabel("y"); ax2.set_zlabel("z")
plt.show()

Why the rotation matters: run the same orbit with a coarse step, twelve steps per turn, for two hundred turns, with Boris and with fourth-order Runge-Kutta. Runge-Kutta is far more accurate per step, but each step leaves the circle by a small amount and the error accumulates in one direction: the kinetic energy drifts steadily. Boris rotates the velocity without changing its length, so the speed is exact at every step however coarse, and only the phase of the orbit is slightly off. For long runs (a particle trapped between magnetic mirrors for a million turns, say) that difference decides which method is usable, and it is the reason notebook 03 uses Boris.

In [ ]:
dt_coarse, n_turns = T / 12, 200
_, vb = it.boris([0, 0, 0], [1e5, 0, 0], e, m_p, [0, 0, 0], [0, 0, 0.01], dt_coarse, 12 * n_turns)
_, vr = it.rk4([0, 0, 0], [1e5, 0, 0], e, m_p, [0, 0, 0], [0, 0, 0.01], dt_coarse, 12 * n_turns)
Eb = np.sum(vb**2, axis=1) / np.sum(vb[0]**2); Er = np.sum(vr**2, axis=1) / np.sum(vr[0]**2)
print(f"after {n_turns} turns at 12 steps per turn: Boris kinetic energy ratio {Eb[-1]:.12f}   RK4 {Er[-1]:.6f}")
turns = np.arange(len(Eb)) / 12
plt.plot(turns, Er - 1, label="RK4"); plt.plot(turns, Eb - 1, label="Boris")
plt.xlabel("turns"); plt.ylabel("kinetic energy / initial - 1"); plt.legend(); plt.title("energy drift at 12 steps per cyclotron period"); plt.show()

### 3.4 Crossed electric and magnetic fields

Now add a uniform $\mathbf{E}$ perpendicular to $\mathbf{B}$.

**Velocity selector.** A particle moving at $\mathbf{v}$ with $q\mathbf{E} = -q\mathbf{v}\times\mathbf{B}$ feels no net force and goes straight. For $\mathbf{E} = E\hat{\mathbf{x}}$, $\mathbf{B} = B\hat{\mathbf{z}}$ and $\mathbf{v} = -v\hat{\mathbf{y}}$: $\mathbf{v}\times\mathbf{B} = -vB\,(\hat{\mathbf{y}}\times\hat{\mathbf{z}}) = -vB\hat{\mathbf{x}}$, which cancels $E\hat{\mathbf{x}}$ when $E = vB$, that is $v = E/B$, independent of the charge and the mass. (A particle moving along $+y$ instead has the magnetic force along $+x$, the same way as the electric force, and is deflected more, not less: the undeflected direction is the direction of $\mathbf{E}\times\mathbf{B}$.) Particles of any other speed are deflected. This is how a beam of known speed is prepared.

**The $\mathbf{E}\times\mathbf{B}$ drift.** What does a particle that starts from rest do? Move to a frame travelling with constant velocity $\mathbf{u}$. At speeds far below $c$ the magnetic field is the same in the new frame and the electric field becomes $\mathbf{E}' = \mathbf{E} + \mathbf{u}\times\mathbf{B}$ (the force $q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$ written in terms of $\mathbf{v}' = \mathbf{v} - \mathbf{u}$). Choose $\mathbf{u} = \mathbf{E}\times\mathbf{B}/B^2$. Then $\mathbf{u}\times\mathbf{B} = (\mathbf{E}\times\mathbf{B})\times\mathbf{B}/B^2 = -\mathbf{E}$ when $\mathbf{E}\perp\mathbf{B}$ (use the BAC-CAB rule of primer M1.0a), so $\mathbf{E}' = 0$. In the moving frame the particle sees only $\mathbf{B}$ and circles. Back in the laboratory the circle drifts along at
$$\mathbf{v}_{\text{drift}} = \frac{\mathbf{E}\times\mathbf{B}}{B^2},$$
the same for every charge, every mass and every sign: electrons and protons drift together. The path is a cycloid, the curve traced by a point on a rolling wheel.

**Worked example 7.** $E = 100\,$V/m along $x$, $B = 0.01\,$T along $z$. $\mathbf{E}\times\mathbf{B} = EB\,(\hat{\mathbf{x}}\times\hat{\mathbf{z}}) = -EB\hat{\mathbf{y}}$, so the drift is $E/B = 10^4\,$m/s along $-y$. A proton starting from rest accelerates along $+x$, is turned by $\mathbf{B}$, comes to rest again after one cyclotron period a distance $2\pi\,v_{\text{drift}}/\omega$ further along $-y$, and repeats. For the velocity selector with the same fields the transmitted speed is also $E/B = 10^4\,$m/s, along $-y$, the direction of $\mathbf{E}\times\mathbf{B}$.

**Magnetic mirrors.** When $\mathbf{B}$ is not uniform but grows along its own direction, the helical motion of section 3.3 is slowed along the field and eventually reversed: the particle is reflected by the region of strong field. The quantity that stays constant is the magnetic moment of the orbit, $mv_\perp^2/2B$, so as $B$ grows $v_\perp$ grows, and since the speed is fixed, $v_\parallel$ must shrink to zero. This is what traps charged particles in the Earth's field and in laboratory plasmas, and notebook 03 shows it with the Boris pusher in the field of two coils.

*The third part of the animation in section 3.3 shows this drift: the proton traces large arches, the electron small ones, and both move down the page at the same measured speed $E/B$.*

In [ ]:
E0, B0 = 100.0, 0.01
x, v = it.boris([0, 0, 0], [0, 0, 0], e, m_p, [E0, 0, 0], [0, 0, B0], T / 200, 200 * 10)
drift = (x[-1] - x[0]) / (10 * T)
print("measured drift velocity:", drift, "m/s    E x B / B^2 =", np.cross([E0, 0, 0], [0, 0, B0]) / B0**2)
xe, _ = it.boris([0, 0, 0], [0, 0, 0], -e, m_e, [E0, 0, 0], [0, 0, B0], 2 * np.pi / it.cyclotron_frequency(e, B0, m_e) / 200, 200 * 10)
print("an electron from rest drifts at:", (xe[-1] - xe[0]) / (10 * 2 * np.pi / it.cyclotron_frequency(e, B0, m_e)), "m/s  (same drift, opposite sense of circling)")

# The velocity selector: v = E/B along -y (the E x B direction) goes straight; other speeds are deflected.
for speed in (0.5e4, 1e4, 2e4):
    xs, _ = it.boris([0, 0, 0], [0, -speed, 0], e, m_p, [E0, 0, 0], [0, 0, B0], T / 200, 200)
    print(f"v = {speed:.0e} m/s along -y: largest sideways excursion during one period = {np.abs(xs[:, 0]).max():.4f} m   (zero only for v = E/B = {E0 / B0:.0e} m/s)")
xs, _ = it.boris([0, 0, 0], [0, +1e4, 0], e, m_p, [E0, 0, 0], [0, 0, B0], T / 200, 200)
print(f"v = 1e+04 m/s along +y: excursion {np.abs(xs[:, 0]).max():.4f} m  (both forces along +x: deflected more, not less)")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x[:, 1], x[:, 0]); ax.set_xlabel("y [m]  (drift direction)"); ax.set_ylabel("x [m]  (E direction)"); ax.set_aspect("equal")
ax.set_title("proton from rest in crossed fields: a cycloid drifting along E x B"); plt.show()

### 3.5 Torque on a current loop and the magnetic dipole moment

Take a rectangular loop with sides $a$ (along $x$) and $b$ (along $y$) in the $xy$ plane, carrying current $I$ counter-clockwise seen from $+z$, in a uniform field $\mathbf{B} = B\hat{\mathbf{x}}$ lying in the plane of the loop. Apply $\mathbf{F} = I\mathbf{L}\times\mathbf{B}$ to each side. The two sides along $x$ are parallel to $\mathbf{B}$ and feel nothing. The side at $x = +a/2$ carries current along $+y$: force $Ib\,(\hat{\mathbf{y}}\times\hat{\mathbf{x}})B = -IbB\hat{\mathbf{z}}$. The side at $x = -a/2$ carries current along $-y$: force $+IbB\hat{\mathbf{z}}$. Equal and opposite forces a distance $a$ apart: no net force, but a torque of magnitude $IbB\cdot a = IAB$ about the $y$ axis, with $A = ab$ the area. The torque turns the loop so that its normal (by the right-hand rule, along $+z$ for this current) lines up with $\mathbf{B}$. For a general angle $\theta$ between the normal and the field the lever arm is $a\sin\theta$ and the torque is $IAB\sin\theta$. All of this is summarised by defining the magnetic dipole moment
$$\mathbf{m} = I\mathbf{A} = IA\,\hat{\mathbf{n}}, \qquad \boldsymbol{\tau} = \mathbf{m}\times\mathbf{B}, \qquad U = -\mathbf{m}\cdot\mathbf{B},$$
where the energy follows from integrating the torque over the angle, and is lowest when $\mathbf{m}$ is aligned with $\mathbf{B}$. The result holds for a loop of any shape (any planar loop can be tiled by small rectangles whose internal sides cancel, the same argument as the integral theorems of primer M1.0d) and for a coil of $N$ turns, which has $\mathbf{m} = NI\mathbf{A}$. Compare the electric dipole: $\boldsymbol{\tau} = \mathbf{p}\times\mathbf{E}$, $U = -\mathbf{p}\cdot\mathbf{E}$, primer M3.0a.

**Worked example 8.** A coil of $100$ turns, $2\,$cm by $3\,$cm, carrying $0.5\,$A: $m = 100\cdot0.5\cdot6\times10^{-4} = 0.03\,$A m$^2$. In a $0.1\,$T field with the normal at $90^\circ$ to the field the torque is $0.03\cdot0.1 = 3\times10^{-3}\,$N m; at $30^\circ$ it is $1.5\times10^{-3}\,$N m. Turning the coil from aligned to anti-aligned costs $2mB = 6\times10^{-3}\,$J.

**Worked example 9.** A charge $q$ circling at speed $v$ on a circle of radius $r$ is a current $I = q/T = qv/2\pi r$ around an area $\pi r^2$, so $m = IA = qvr/2$. For an electron on the smallest Bohr orbit, $r = 5.29\times10^{-11}\,$m and $v = 2.19\times10^6\,$m/s: $m = 1.602\times10^{-19}\cdot2.19\times10^6\cdot5.29\times10^{-11}/2 = 9.28\times10^{-24}\,$A m$^2$, the Bohr magneton, the natural unit of atomic magnetic moments. Chapter 13 adds up moments like this one over all the atoms in a volume and calls the result the magnetization $\mathbf{M}$, the magnetic twin of the polarization $\mathbf{P}$.

In [ ]:
# Worked example 8: torque on a 100-turn coil, directly from I dl x B on the four sides.
N, I, a, b, B0 = 100, 0.5, 0.02, 0.03, 0.1
m_vec = N * I * a * b * np.array([0, 0, 1.0])
for theta_deg in (90, 30, 0):
    th = np.radians(theta_deg)
    B = B0 * np.array([np.sin(th), 0, np.cos(th)])                  # angle theta between B and the loop normal z
    corners = np.array([[a / 2, -b / 2, 0], [a / 2, b / 2, 0], [-a / 2, b / 2, 0], [-a / 2, -b / 2, 0]])
    mid, dl = bs.segments(corners)                                   # counter-clockwise seen from +z
    forces = N * I * np.cross(dl, B)
    torque = np.cross(mid, forces).sum(axis=0)
    print(f"theta = {theta_deg:2d} deg: net force {forces.sum(axis=0)} N, torque {torque} N m,  m x B = {np.cross(m_vec, B)},  |m| B sin(theta) = {N * I * a * b * B0 * np.sin(th):.2e}")
print(f"energy difference between anti-aligned and aligned: 2 m B = {2 * N * I * a * b * B0:.2e} J")
print(f"Bohr magneton from an orbiting electron: {e * 2.19e6 * 5.29e-11 / 2:.3e} A m^2")

### 3.6 The Biot-Savart law

A steady current $I$ along a wire produces at the point $\mathbf{r}$ the field
$$\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi}\oint\frac{I\,d\mathbf{l}'\times\hat{\mathbf{R}}}{R^2} = \frac{\mu_0}{4\pi}\oint\frac{I\,d\mathbf{l}'\times\mathbf{R}}{R^3}, \qquad \mathbf{R} = \mathbf{r} - \mathbf{r}',$$
where $d\mathbf{l}'$ is a step along the wire in the direction of the current at the source point $\mathbf{r}'$, and the integral runs around the whole circuit (a steady current has to flow in closed loops). The constant $\mu_0 = 4\pi\times10^{-7}\,$T m/A $= 1.2566\times10^{-6}\,$T m/A; the $4\pi$ in its definition cancels the $4\pi$ in the law. Direction: by the right-hand rule for $d\mathbf{l}'\times\hat{\mathbf{R}}$, which for a straight wire means thumb along the current, fingers curling in the direction of $\mathbf{B}$ around it. Each element contributes an inverse-square field, like a point charge, but the cross product makes the contribution zero along the line of the element and largest at right angles to it. On the computer the integral is a sum: cut the wire into straight segments, evaluate $d\mathbf{l}\times\mathbf{R}/R^3$ from each segment midpoint to every observation point, add, and multiply by $\mu_0I/4\pi$. That is `emlib.biot_savart.wire_B`, with the segments supplied by `segments`, `loop`, `straight_wire` and `helix`.

**Worked example 10 (the infinite straight wire).** Current $I$ along the $z$ axis; find $\mathbf{B}$ at distance $r$ from it, say at $(r, 0, 0)$. An element $dz$ at height $z$ has $\mathbf{R} = (r, 0, -z)$, $R = \sqrt{r^2 + z^2}$, and $d\mathbf{l}\times\mathbf{R} = dz\,\hat{\mathbf{z}}\times(r, 0, -z) = r\,dz\,\hat{\mathbf{y}}$. Every element contributes along $+\hat{\mathbf{y}}$, which circulates around the wire as the right-hand rule says. So
$$B_y = \frac{\mu_0I}{4\pi}\int_{-\infty}^{\infty}\frac{r\,dz}{(r^2 + z^2)^{3/2}} = \frac{\mu_0I}{4\pi}\cdot r\cdot\frac{2}{r^2} = \frac{\mu_0I}{2\pi r},$$
using $\int_{-\infty}^\infty dz/(r^2 + z^2)^{3/2} = \big[z/(r^2\sqrt{r^2 + z^2})\big]_{-\infty}^\infty = 2/r^2$. The field falls as $1/r$, as the line charge's electric field did, and for the same reason: the far parts of an infinite line keep contributing. For a segment from $-L$ to $L$ the same integral gives $2L/(r^2\sqrt{r^2 + L^2})$, so $B = \dfrac{\mu_0I}{4\pi r}\dfrac{2L}{\sqrt{r^2 + L^2}}$ on the bisector; it approaches the infinite-wire value when $L \gg r$. With $I = 10\,$A at $r = 2\,$cm: $B = 2\times10^{-7}\cdot10/0.02 = 10^{-4}\,$T. A segment of half-length $2\,$cm gives $1/\sqrt2$ of that, $7.07\times10^{-5}\,$T.

**Worked example 11 (the axis of a circular loop).** A loop of radius $R$ in the $xy$ plane, centred on the origin, current counter-clockwise seen from $+z$; find $\mathbf{B}$ at $(0, 0, z)$. Every element $d\mathbf{l}$ is tangent to the circle and $\mathbf{R}$ runs from the element to the axis point, so $d\mathbf{l}\perp\mathbf{R}$ and $|d\mathbf{l}\times\mathbf{R}| = dl\,R$ with $R = \sqrt{R^2 + z^2}$ (the same letter is used for the loop radius and the distance; the context distinguishes them). The contribution $d\mathbf{B}$ is perpendicular to $\mathbf{R}$ and tilted off the axis by the angle $\alpha$ with $\cos\alpha = R_{\text{loop}}/\sqrt{R_{\text{loop}}^2 + z^2}$. Going once around the loop, the components perpendicular to the axis point in every direction and cancel; the components along the axis all add. So
$$B_z = \frac{\mu_0I}{4\pi}\oint\frac{dl}{R_{\text{loop}}^2 + z^2}\cos\alpha = \frac{\mu_0I}{4\pi}\cdot\frac{2\pi R_{\text{loop}}\cdot R_{\text{loop}}}{(R_{\text{loop}}^2 + z^2)^{3/2}} = \frac{\mu_0IR^2}{2(R^2 + z^2)^{3/2}},$$
with $R$ the loop radius again. At the centre, $B = \mu_0I/2R$. For $R = 5\,$cm and $I = 2\,$A: centre $1.2566\times10^{-6}\cdot2/0.1 = 2.513\times10^{-5}\,$T; at $z = R$ the factor $(R^2 + z^2)^{3/2} = (2R^2)^{3/2} = 2\sqrt2R^3$ gives $2.513\times10^{-5}/2\sqrt2 = 8.886\times10^{-6}\,$T; at $z = 10R = 0.5\,$m, $(0.0025 + 0.25)^{3/2} = 0.1269$, so $B = 1.2566\times10^{-6}\cdot2\cdot0.0025/0.1269 = 2.476\times10^{-8}\,$T, three orders down and already close to the $1/z^3$ law of the next section.

The code checks all three closed forms against the segment sum and draws the loop's field in a plane through its axis.

In [ ]:
# Straight wire: infinite (long) and finite.
I = 10.0
mid, dl = bs.segments(bs.straight_wire([0, 0, -50], [0, 0, 50], 100000), closed=False)
r = np.array([0.02, 0.05, 0.2])
B = bs.wire_B(mid, dl, I, np.stack([r, 0 * r, 0 * r], axis=-1))
print("long wire, point at r = 2 cm on +x:  B =", sci(B[0]), " (along +y: circulating)")
print("  |B| from segments:", np.linalg.norm(B, axis=1), "   mu0 I / 2 pi r:", bs.straight_wire_B(I, r))
midf, dlf = bs.segments(bs.straight_wire([0, 0, -0.02], [0, 0, 0.02], 2000), closed=False)
Bf = bs.wire_B(midf, dlf, I, np.array([0.02, 0, 0]))
print(f"segment of half-length 2 cm at r = 2 cm: {np.linalg.norm(Bf):.4e} T   closed form {bs.finite_wire_B(I, 0.02, 0.02):.4e} T   (1/sqrt2 of the infinite wire)")

# Circular loop on its axis.
R, I = 0.05, 2.0
mid, dl = bs.segments(bs.loop(R, 400))
z = np.array([0.0, R, 10 * R])
Bz = bs.wire_B(mid, dl, I, np.stack([0 * z, 0 * z, z], axis=-1))[:, 2]
print("\nloop R = 5 cm, I = 2 A, on the axis at z = 0, R, 10R:")
print("  segments:   ", sci(Bz)); print("  closed form:", sci(bs.loop_axis_B(R, I, z)))

# Field map in the xz plane through the axis.
s = np.linspace(-0.15, 0.15, 31)
X, Z = np.meshgrid(s, s, indexing="ij")
P = np.stack([X, np.zeros_like(X), Z], axis=-1)
Bmap = bs.wire_B(mid, dl, I, P)
Bm = np.linalg.norm(Bmap, axis=-1)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.streamplot(s, s, Bmap[..., 0].T, Bmap[..., 2].T, color=np.log10(Bm).T, cmap="viridis", density=1.4)
ax.plot([-R, R], [0, 0], "o", color="C3", ms=8, label="wire (current into / out of the page)")
ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]"); ax.legend(loc="upper right"); ax.set_title("field of a circular loop in a plane through its axis (colour = log10 |B|)")
plt.show()

### 3.7 The loop as a magnetic dipole

Far from the loop, $z \gg R$, the axial field becomes $B_z \approx \mu_0IR^2/2z^3 = \dfrac{\mu_0}{4\pi}\dfrac{2m}{z^3}$ with $m = I\pi R^2$. That is exactly the on-axis field of a dipole. The full far field, at any direction, is
$$\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi}\,\frac{3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m}}{r^3}, \qquad \mathbf{m} = I\mathbf{A},$$
the same expression as the electric dipole field of primer M1.0e with $\mathbf{p}\to\mathbf{m}$ and $1/4\pi\epsilon_0\to\mu_0/4\pi$. On the axis it is $2m$ over $r^3$ along $\mathbf{m}$; in the equatorial plane it is $m$ over $r^3$ against $\mathbf{m}$, half as strong and reversed. The correspondence is not an accident: outside their sources, both fields are curl-free and divergence-free, so they satisfy the same equations and have the same multipole expansion. The difference is inside: electric dipole field lines run from $+$ to $-$ through the dipole, magnetic ones continue through the loop and close, so the fields near the source point opposite ways. For a loop of $N$ turns, or a coil, $\mathbf{m} = NI\mathbf{A}$; a bar magnet is, from outside, a dipole too, with $\mathbf{m}$ from its south pole to its north.

**Worked example 12.** The loop of worked example 11 has $m = 2\cdot\pi\cdot0.05^2 = 1.571\times10^{-2}\,$A m$^2$. The dipole formula on the axis at $z = 0.5\,$m gives $10^{-7}\cdot2\cdot1.571\times10^{-2}/0.125 = 2.513\times10^{-8}\,$T, within $1.5\%$ of the exact $2.476\times10^{-8}\,$T found above (the next term in the expansion is of relative size $\tfrac32R^2/z^2 = 1.5\%$). In the equatorial plane at $r = 0.5\,$m: $B = -1.257\times10^{-8}\,\hat{\mathbf{z}}\,$T.

In [ ]:
m_vec = np.array([0, 0, I * np.pi * R**2])
print(f"dipole moment of the loop: {m_vec[2]:.4e} A m^2")
obs = np.array([[0, 0, 0.5], [0.5, 0, 0], [0.3, 0.3, 0.3], [0, 0, 0.1]])
B_loop = bs.wire_B(mid, dl, I, obs); B_dip = bs.dipole_B(m_vec, obs)
for p, bl, bd in zip(obs, B_loop, B_dip):
    print(f"at {p} (r = {np.linalg.norm(p) / R:4.1f} R): loop {sci(bl)}   dipole {sci(bd)}   ratio of magnitudes {np.linalg.norm(bl) / np.linalg.norm(bd):.4f}")
print("the dipole law is good to about 3R^2/(2r^2): fine at 10R, poor at 2R")

### Exercise: the centre of a square loop

Replace the body of `square_loop_centre_B` so that it returns $|\mathbf{B}|$ at the centre of a square loop of side $a$ carrying current $I$, using the finite-segment result of worked example 10 for each of the four sides (each side is a segment of half-length $a/2$ at distance $a/2$ from the centre, and all four contribute in the same direction). The closed form is $2\sqrt2\,\mu_0I/\pi a$. The check compares with the segment sum over the four sides.

In [ ]:
def square_loop_centre_B(a, I):
    '''|B| at the centre of a square loop of side a carrying current I.'''
    one_side = bs.finite_wire_B(I, a / 2, a / 2)                    # replace from here with your own version
    return 4 * one_side


for a, I in ((0.1, 1.0), (0.02, 5.0)):
    corners = np.array([[a / 2, -a / 2, 0], [a / 2, a / 2, 0], [-a / 2, a / 2, 0], [-a / 2, -a / 2, 0]])
    pts = np.vstack([bs.straight_wire(corners[k], corners[(k + 1) % 4], 500)[:-1] for k in range(4)])
    mid_sq, dl_sq = bs.segments(pts)
    B_num = np.linalg.norm(bs.wire_B(mid_sq, dl_sq, I, np.array([0.0, 0.0, 0.0])))
    print(f"a = {a} m, I = {I} A:  yours {square_loop_centre_B(a, I):.5e} T   segments {B_num:.5e} T   closed form {2 * np.sqrt(2) * MU0 * I / (np.pi * a):.5e} T")
    assert np.isclose(square_loop_centre_B(a, I), B_num, rtol=1e-4)
print("square loop exercise passed")

## 4. Verify

Three closed forms against the library, each with an assert: the straight wire ($\mu_0I/2\pi r$), the loop on its axis, and the Larmor radius of a Boris orbit. These are the same checks that `tests/test_biot_savart.py` and `tests/test_integrators.py` run.

In [ ]:
I = 3.0
mid, dl = bs.segments(bs.straight_wire([0, 0, -50], [0, 0, 50], 100000), closed=False)
r = np.array([0.1, 0.5, 2.0])
B_wire = bs.wire_B(mid, dl, I, np.stack([r, 0 * r, 0 * r], axis=-1))
print("straight wire, max relative error:", np.abs(B_wire[:, 1] / bs.straight_wire_B(I, r) - 1).max())
assert np.allclose(B_wire[:, 1], bs.straight_wire_B(I, r), rtol=1e-3)

R, I = 0.3, 2.0
mid, dl = bs.segments(bs.loop(R, 400))
z = np.linspace(0, 2.0, 9)
B_loop = bs.wire_B(mid, dl, I, np.stack([0 * z, 0 * z, z], axis=-1))
print("loop on axis, max relative error:", np.abs(B_loop[:, 2] / bs.loop_axis_B(R, I, z) - 1).max(), "  transverse components:", np.abs(B_loop[:, :2]).max())
assert np.allclose(B_loop[:, 2], bs.loop_axis_B(R, I, z), rtol=1e-5)

B0, v0 = 0.01, 1e5
T = 2 * np.pi / it.cyclotron_frequency(e, B0, m_p)
x, v = it.boris([0, 0, 0], [v0, 0, 0], e, m_p, [0, 0, 0], [0, 0, B0], T / 400, 400)
r_orbit = np.linalg.norm(x - x.mean(axis=0), axis=1).mean()
print(f"Boris orbit radius {r_orbit:.5f} m   Larmor radius {it.larmor_radius(m_p, v0, e, B0):.5f} m")
assert np.isclose(r_orbit, it.larmor_radius(m_p, v0, e, B0), rtol=1e-3)
print("all checks passed")

## 5. Explore and rebuild

Explore. Two long parallel wires $2\,$cm apart carry equal currents in opposite directions (a transmission line). Map $|\mathbf{B}|$ in the plane perpendicular to the wires. Between the wires the two fields add; outside they cancel more and more. Sample $|\mathbf{B}|$ along the line through both wires from $1\,$cm beyond one wire to $1\,$m, fit the fall-off exponent far away, and compare with the $1/r$ of a single wire. Then do the same for a circular loop along its axis: from $\mu_0I/2R$ at the centre to $\mu_0IR^2/2z^3$ far away, and find where the dipole law becomes accurate to $1\%$.

Rebuild. Close this notebook. On paper, derive the field of an infinite straight wire and the on-axis field of a circular loop from the Biot-Savart law (worked examples 10 and 11), and the force per length between parallel wires. Then write the cyclotron frequency, the Larmor radius and the $\mathbf{E}\times\mathbf{B}$ drift from the force law alone. In a blank editor, write the Biot-Savart segment sum from scratch (segments, midpoints, $d\mathbf{l}\times\mathbf{R}/R^3$, broadcast over observation points) and reproduce the loop-axis check.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| Lorentz force | $\mathbf{F} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$ | always (point charge) |
| magnetic work | $\mathbf{F}_{\text{mag}}\cdot\mathbf{v} = 0$: speed unchanged by $\mathbf{B}$ | always |
| force on a current | $d\mathbf{F} = I\,d\mathbf{l}\times\mathbf{B}$; straight wire $\mathbf{F} = I\mathbf{L}\times\mathbf{B}$ | $d\mathbf{l}$ along the current; uniform $\mathbf{B}$ for the second form |
| parallel wires | $F/L = \mu_0I_1I_2/2\pi d$, attractive for parallel currents | long wires |
| Hall effect | $V_H = IB/nqt$; sign gives the sign of the carriers, size gives $n$ | flat strip, $\mathbf{B}$ normal to it |
| circular orbit | $r = mv_\perp/\lvert q\rvert B$, $\omega = \lvert q\rvert B/m$, $T = 2\pi m/\lvert q\rvert B$ | uniform $\mathbf{B}$, no $\mathbf{E}$ |
| sense of rotation | positive charge clockwise seen looking against $\mathbf{B}$; negative charge the other way | |
| helix pitch | $v_\parallel T$ | $v_\parallel$ constant |
| velocity selector | straight through when $v = E/B$ along $\mathbf{E}\times\mathbf{B}$ | $\mathbf{E}\perp\mathbf{B}\perp\mathbf{v}$ |
| $\mathbf{E}\times\mathbf{B}$ drift | $\mathbf{v}_d = \mathbf{E}\times\mathbf{B}/B^2$, independent of $q$ and $m$ | $\mathbf{E}\perp\mathbf{B}$, $E \ll cB$ |
| magnetic mirror | $mv_\perp^2/2B$ constant; particle reflected where $B$ is large | slowly varying $\mathbf{B}$ |
| magnetic moment | $\mathbf{m} = NI\mathbf{A}$; orbiting charge $m = qvr/2$ | planar loop |
| torque and energy | $\boldsymbol{\tau} = \mathbf{m}\times\mathbf{B}$, $U = -\mathbf{m}\cdot\mathbf{B}$ | uniform $\mathbf{B}$ |
| Biot-Savart | $\mathbf{B} = \dfrac{\mu_0}{4\pi}\displaystyle\oint\dfrac{I\,d\mathbf{l}'\times\mathbf{R}}{R^3}$, $\mu_0 = 4\pi\times10^{-7}\,$T m/A | steady current in a closed circuit |
| straight wire | $B = \mu_0I/2\pi r$, circulating by the right-hand rule | infinite wire |
| finite segment | $B = \dfrac{\mu_0I}{4\pi r}\dfrac{2L}{\sqrt{r^2 + L^2}}$ | on the perpendicular bisector |
| loop on axis | $B_z = \dfrac{\mu_0IR^2}{2(R^2 + z^2)^{3/2}}$; centre $\mu_0I/2R$ | circular loop |
| square loop centre | $B = 2\sqrt2\,\mu_0I/\pi a$ | side $a$ |
| loop far field | $\mathbf{B} = \dfrac{\mu_0}{4\pi}\dfrac{3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m}}{r^3}$, $\mathbf{m} = I\pi R^2\hat{\mathbf{n}}$ | $r \gg R$ |
| Boris pusher | half kick, exact rotation about $\mathbf{B}$, half kick, then move | conserves speed in pure $\mathbf{B}$ for any step |

## 7. Warm-up answers

1. $\mathbf{F} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$. Only the electric part can change the speed; the magnetic part is perpendicular to $\mathbf{v}$ and does no work.
2. $\hat{\mathbf{x}}\times\hat{\mathbf{z}} = -\hat{\mathbf{y}}$, so the force on the proton is along $-y$. The electron, with the opposite sign of charge, is pushed along $+y$.
3. The magnetic force has constant magnitude $qvB$ and is always perpendicular to the velocity, which is precisely a centripetal force. $r = mv/|q|B$ and $T = 2\pi m/|q|B$; the period does not depend on the speed.
4. $F/L = \mu_0I_1I_2/2\pi d$, attractive for currents in the same direction, repulsive for opposite currents.
5. $d\mathbf{B} = (\mu_0/4\pi)\,I\,d\mathbf{l}\times\hat{\mathbf{R}}/R^2$. The current element $I\,d\mathbf{l}$ plays the role of the charge, the inverse square is the same, and the cross product with $\hat{\mathbf{R}}$ replaces $\hat{\mathbf{R}}$ itself, making the field circulate around the current instead of pointing away from it.
6. Every element is at distance $R$ from the centre with $d\mathbf{l}\perp\mathbf{R}$, so each contributes $(\mu_0I/4\pi)\,dl/R^2$ along the axis, and summing $dl$ to $2\pi R$ gives $B = \mu_0I/2R$.
7. A dipole field, $\mathbf{B} = (\mu_0/4\pi)(3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m})/r^3$, characterised by the magnetic moment $\mathbf{m} = I\mathbf{A}$.